In [1]:
#setup & constants
import numpy as np
from datetime import datetime

# Path configuration for Google Colab
FILE_PATH = "/content/hostel_bois.txt"

In [3]:
#chat parser function
def parse_chat_file(file_path):
    parsed_messages = []
    system_msg_count = 0
    media_omitted_count = 0
    deleted_msg_count = 0
    media_per_person = {}
    deleted_per_person = {}

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.read().split('\n')
    except FileNotFoundError:
        print(f"Error: Could not find file at {file_path}. Please upload 'hostel_bois.txt' to Colab.")
        return [], 0, 0, 0, {}, {}

    for line in lines:
        line = line.strip()
        if not line:
            continue

        is_new_message = False
        if len(line) >= 8:
            parts = line.split(', ')
            if len(parts) > 1 and len(parts[0]) == 8:
                date_part = parts[0]
                if date_part[2] == '/' and date_part[5] == '/' and date_part.replace('/', '').isdigit():
                    is_new_message = True

        if not is_new_message:
            if parsed_messages:
                parsed_messages[-1]['text'] += " " + line
            continue

        parts_split = line.split(' ', 1)
        timestamp_str = parts_split[0].rstrip(',')
        body = parts_split[1] if len(parts_split) > 1 else ""

        time_str = body.split(' ', 1)[0]
        full_timestamp = f"{timestamp_str}, {time_str}"

        remaining_text = body.split(' ', 1)[1] if len(body.split(' ', 1)) > 1 else ""

        if '-' in remaining_text:
            remaining_text = remaining_text.split('-', 1)[1].strip()

        if ':' not in remaining_text:
            system_msg_count += 1
            continue

        sender, message_text = remaining_text.split(':', 1)
        sender = sender.strip()
        message_text = message_text.strip()

        if sender not in media_per_person: media_per_person[sender] = 0
        if sender not in deleted_per_person: deleted_per_person[sender] = 0

        if message_text == "<Media omitted>":
            media_omitted_count += 1
            media_per_person[sender] += 1
            parsed_messages.append({
                'timestamp': full_timestamp, 'sender': sender, 'text': message_text,
                'is_real_text': False, 'is_deleted': False
            })
            continue

        if message_text == "This message was deleted":
            deleted_msg_count += 1
            deleted_per_person[sender] += 1
            parsed_messages.append({
                'timestamp': full_timestamp, 'sender': sender, 'text': message_text,
                'is_real_text': False, 'is_deleted': True
            })
            continue

        parsed_messages.append({
            'timestamp': full_timestamp, 'sender': sender, 'text': message_text,
            'is_real_text': True, 'is_deleted': False
        })

    print(f"Successfully parsed {len(parsed_messages)} messages.")
    return parsed_messages, system_msg_count, media_omitted_count, deleted_msg_count, media_per_person, deleted_per_person

# Execute the parser
messages, sys_cnt, media_cnt, del_cnt, media_by_user, del_by_user = parse_chat_file(FILE_PATH)

Successfully parsed 3174 messages.


In [4]:
#Basic Aggregations & Group Trends
def process_group_metrics(messages):
    if not messages: return
    user_counts = {}
    busiest_day_map = {}
    busiest_hour_map = {}
    unique_days = set()

    for m in messages:
        sender = m['sender']
        user_counts[sender] = user_counts.get(sender, 0) + 1

        dt_obj = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')
        day_str = dt_obj.strftime('%d %B %Y')
        hour_str = dt_obj.strftime('%H:00')

        unique_days.add(dt_obj.strftime('%Y-%m-%d'))
        busiest_day_map[day_str] = busiest_day_map.get(day_str, 0) + 1
        busiest_hour_map[hour_str] = busiest_hour_map.get(hour_str, 0) + 1

    sorted_users = sorted(user_counts.items(), key=lambda x: x[1], reverse=True)
    busiest_day = max(busiest_day_map.items(), key=lambda x: x[1])
    busiest_hour = max(busiest_hour_map.items(), key=lambda x: x[1])
    avg_peak_hour = busiest_hour[1] / len(unique_days) if unique_days else 0

    return sorted_users, busiest_day, busiest_hour, len(unique_days), avg_peak_hour

sorted_users, active_day, active_hour, total_days_range, avg_peak_hour = process_group_metrics(messages)

In [5]:
#NumPy Heatmap Matrix
def generate_heatmap_matrix(messages):
    participants = sorted(list(set(m['sender'] for m in messages)))
    user_to_idx = {user: idx for idx, user in enumerate(participants)}
    heatmap_matrix = np.zeros((len(participants), 24), dtype=int)

    for m in messages:
        sender = m['sender']
        dt_obj = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')
        heatmap_matrix[user_to_idx[sender], dt_obj.hour] += 1

    return heatmap_matrix, participants

heatmap, user_list = generate_heatmap_matrix(messages)

In [6]:
#Word Frequency Counter
def compute_word_frequencies(messages):
    word_counts = {}
    STOP_WORDS = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'was', 'with', 'you', 'my', 'bhai', 'hai', 'na', 'yaar'}
    PUNCTUATION = '.,?!:;"\'()[]{}<>-'

    for m in messages:
        if not m['is_real_text']: continue
        for word in m['text'].lower().split():
            word = word.strip(PUNCTUATION)
            if word and word not in STOP_WORDS:
                word_counts[word] = word_counts.get(word, 0) + 1

    return sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:5]

top_words = compute_word_frequencies(messages)

In [8]:
#Gaps & Silent Streaks
def compute_response_and_streaks(messages, user_list):
    response_gaps = {user: [] for user in user_list}
    for i in range(1, len(messages)):
        prev_m, curr_m = messages[i-1], messages[i]
        if prev_m['sender'] != curr_m['sender']:
            prev_time = datetime.strptime(prev_m['timestamp'], '%d/%m/%y, %H:%M')
            curr_time = datetime.strptime(curr_m['timestamp'], '%d/%m/%y, %H:%M')
            gap = (curr_time - prev_time).total_seconds()
            if gap >= 0: response_gaps[curr_m['sender']].append(gap)

    avg_responses = {u: (sum(gats)/len(gats) if gats else 0.0) for u, gats in response_gaps.items()}

    all_dates = sorted(list(set(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M').strftime('%Y-%m-%d') for m in messages)))
    user_active_dates = {user: set(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M').strftime('%Y-%m-%d') for m in messages if m['sender'] == user) for user in user_list}

    longest_streaks = {}
    for user in user_list:
        max_s, curr_s = 0, 0
        for d in all_dates:
            if d not in user_active_dates[user]:
                curr_s += 1
                max_s = max(max_s, curr_s)
            else:
                curr_s = 0
        longest_streaks[user] = max_s

    return avg_responses, longest_streaks

avg_resp_times, silent_streaks = compute_response_and_streaks(messages, user_list)

In [10]:
#Personality Profiler Archetypes
def calculate_archetypes(messages, user_list, heatmap):
    archetypes = {}
    user_to_idx = {user: idx for idx, user in enumerate(user_list)}

    user_bursts = {user: [] for user in user_list}
    curr_s, curr_cnt = messages[0]['sender'] if messages else None, 1
    for i in range(1, len(messages)):
        if messages[i]['sender'] == curr_s: curr_cnt += 1
        else:
            if curr_s: user_bursts[curr_s].append(curr_cnt)
            curr_s, curr_cnt = messages[i]['sender'], 1
    if curr_s: user_bursts[curr_s].append(curr_cnt)
    avg_bursts = {u: (sum(b)/len(b) if b else 0) for u, b in user_bursts.items()}

    for user in user_list:
        r_idx = user_to_idx[user]
        total_user_msgs = np.sum(heatmap[r_idx, :])
        night_msgs = np.sum(heatmap[r_idx, 23:]) + np.sum(heatmap[r_idx, 0:5])
        night_pct = (night_msgs / total_user_msgs * 100) if total_user_msgs > 0 else 0

        user_texts = [m['text'] for m in messages if m['sender'] == user and m['is_real_text']]
        avg_word_len = (sum(len(msg.split()) for msg in user_texts) / len(user_texts)) if user_texts else 0
        caps_pct = (sum(1 for m in user_texts if m.isupper() and len(m) > 3) / len(user_texts) * 100) if user_texts else 0

        if avg_bursts[user] > 2.5: label, det = "THE SPAMMER", f"avg {avg_bursts[user]:.1f} consecutive msgs"
        elif night_pct > 45.0: label, det = "THE NIGHT OWL", f"{night_pct:.1f}% msgs at night"
        elif avg_word_len > 25.0: label, det = "THE STORYTELLER", f"avg {avg_word_len:.1f} words/msg"
        elif caps_pct > 20.0: label, det = "THE DRAMA QUEEN", f"{caps_pct:.1f}% ALL-CAPS text"
        elif silent_streaks[user] > 4: label, det = "THE GHOST", f"absent for {silent_streaks[user]} consecutive days"
        else: label, det = "THE COMEDIAN", "Keeps the group's vibes balanced"

        archetypes[user] = (label, det)
    return archetypes

archetype_assignments = calculate_archetypes(messages, user_list, heatmap)

In [11]:
#Terminal Report Output Display
def render_terminal_report():
    print("=" * 65)
    print("                 GROUPDNA REPORT — \"Hostel Bois 4ever\"")
    print(f"            {total_days_range} days  •  {len(messages)} messages  •  {len(user_list)} members")
    print("=" * 65)
    print(f" Busiest Day  : {active_day[0]} ({active_day[1]} messages)")
    print(f" Busiest Hour : {active_hour[0]} (avg {int(avg_peak_hour)} messages per day)")
    print("\n MESSAGES PER PERSON")

    max_user_msgs = max(dict(sorted_users).values())
    for user, count in sorted_users:
        pct = (count / len(messages)) * 100
        print(f"  {user:<8} {'█' * int((count / max_user_msgs) * 20):<20} {count} ({pct:.1f}%)")

    print("\n ACTIVITY HEATMAP (Columns: 00 to 23 Hours)")
    for idx, user in enumerate(user_list):
        row_str = "".join(". " if val < 2 else "░ " if val < 5 else "▒ " if val < 10 else "█ " for val in heatmap[idx, :])
        print(f"  {user:<8} {row_str}")

    print("\n PERSONALITY ARCHETYPES")
    for user, profile in archetype_assignments.items():
        print(f"  {user:<8} → {profile[0]} ({profile[1]})")
    print("=" * 65)

render_terminal_report()

                 GROUPDNA REPORT — "Hostel Bois 4ever"
            60 days  •  3174 messages  •  6 members
 Busiest Day  : 04 May 2024 (76 messages)
 Busiest Hour : 18:00 (avg 4 messages per day)

 MESSAGES PER PERSON
  Rahul    ████████████████████ 953 (30.0%)
  Priya    ███████████████      718 (22.6%)
  Neha     █████████████        635 (20.0%)
  Aman     ██████████           490 (15.4%)
  Karan    ███████              354 (11.2%)
  Vikas                         24 (0.8%)

 ACTIVITY HEATMAP (Columns: 00 to 23 Hours)
  Aman     █ █ █ █ █ . . . . . . . . . █ █ █ ▒ █ ▒ █ █ . █ 
  Karan    . . . . . . . ░ █ █ █ █ █ █ █ █ █ █ █ █ █ █ ▒ ▒ 
  Neha     . . . . . █ ░ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
  Priya    . . . . . . █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ ▒ 
  Rahul    ░ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
  Vikas    . . . . . . . . ░ . . . ░ ░ . . . ░ ░ ░ . . . ░ 

 PERSONALITY ARCHETYPES
  Aman     → THE SPAMMER (avg 2.7 consecutive msgs)
  Karan    → THE STORYTELLER (avg 57.0